# 04. 集計とサマリ

03章で作った `fact` 34行を、**日次サマリにまとめる**のがこの章です。

## この章のゴール

```
  fact 34行  26,489円   マスタが付いた明細。除外はまだしていない
     ↓   何を集計に入れるか決める (除外)
  target 32行  25,139円  集計対象
     ↓   日次 × 店舗 × カテゴリ にまとめる
  サマリ    店舗別 30行 / カテゴリ別 28行
```

| 名前 | 中身 | 作る章 |
| --- | --- | --- |
| `fact` | マスタを結合した明細。**除外前** | 03章 |
| `target` | `fact` から集計対象外を除いたもの | **04章(この章)** |

**`fact` と `target` は別のものです。** 行数も金額も違います。
どちらの話をしているかを、いつも意識してください。
この章では、`fact` から `target` を作るところが山場になります。

## この章で決めること

03章の最後に「あとで判断が要るもの」が4つ挙がっていました。

- テスト伝票(1行)を入れるか
- 閉店した大宮店の売上(1行)を入れるか
- 返品(2行)をどう扱うか
- 売上が無かった日をどう出すか

**どれも、データを見ても答えは出ません。** 決めるのは人間です。
この章では、決め方と、決めたことを**コードと出力に残す方法**を扱います。

---
## 0. 前章までのまとめ

**次のセルは01〜03章の答えです。読み飛ばして実行してかまいません。**

In [ ]:
import unicodedata

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

# ---------------- 01章: 取り込み ----------------
COLUMNS = ["sale_date", "shop_name", "item_cd", "qty", "amount",
           "tax_type", "note", "source"]

RENAME_STORE = {"売上日": "sale_date", "店舗名": "shop_name", "商品CD": "item_cd",
                "数量": "qty", "金額": "amount", "備考": "note"}
RENAME_EC = {"受注日": "sale_date", "店舗名": "shop_name", "商品CD": "item_cd",
             "数量": "qty", "金額税抜": "amount", "ステータス": "note"}

SOURCES = [
    {"path": "/data/sales_2024-04_old.csv", "encoding": "cp932",
     "tax_type": "税込", "source": "old", "rename": RENAME_STORE},
    {"path": "/data/sales_2024-04_new.csv", "encoding": "utf-8",
     "tax_type": "税込", "source": "new", "rename": RENAME_STORE},
    {"path": "/data/sales_2024-04_ec.csv", "encoding": "utf-8",
     "tax_type": "税抜", "source": "ec", "rename": RENAME_EC},
]


def read_one(spec):
    df = pd.read_csv(spec["path"], dtype=str, keep_default_na=False,
                     encoding=spec["encoding"])
    df = df.rename(columns=spec["rename"])
    df = df.assign(tax_type=spec["tax_type"], source=spec["source"])
    return df[COLUMNS]


def load_raw(sources=SOURCES):
    df = pd.concat([read_one(s) for s in sources], ignore_index=True)
    print(f"取り込み: {len(df)}行  内訳 {df['source'].value_counts().to_dict()}")
    return df


# ---------------- 02章: クレンジング ----------------
NA_TOKENS = ["", "-", "N/A"]
TEXT_COLUMNS = ["sale_date", "shop_name", "item_cd", "qty", "amount", "note"]
FORMATS = ["%Y/%m/%d", "%Y年%m月%d日", "%Y-%m-%d"]
TAX_RATE = 1.1


def norm(s):
    """NFKC正規化して、前後の空白を落とす。"""
    return unicodedata.normalize("NFKC", s).strip()


def parse_date(s):
    d = pd.to_datetime(s, format=FORMATS[0], errors="coerce")
    for fmt in FORMATS[1:]:
        d = d.fillna(pd.to_datetime(s, format=fmt, errors="coerce"))
    return d


def clean_raw(raw):
    """raw を整形して (clean, rejected) に分ける。行は1つも捨てない。"""
    df = raw.copy()
    for c in TEXT_COLUMNS:
        df[c] = df[c].map(norm)
    df = df.replace(NA_TOKENS, pd.NA)
    df["item_cd"] = df["item_cd"].str.zfill(4)
    df["amount"] = df["amount"].str.replace(r"[¥,]", "", regex=True)
    df["sale_date"] = parse_date(df["sale_date"])
    df["qty"] = pd.to_numeric(df["qty"], errors="coerce").astype("Int64")
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce").astype("Int64")
    is_excl = df["tax_type"] == "税抜"
    df["amount_incl"] = df["amount"].where(
        ~is_excl, (df["amount"] * TAX_RATE).round()).astype("Int64")
    ok = df["sale_date"].notna() & df["qty"].notna() & df["amount"].notna()
    clean = df[ok].reset_index(drop=True)
    rejected = df[~ok].reset_index(drop=True)
    rate = len(rejected) / len(df) * 100
    print(f"クレンジング: {len(df)}行 → clean {len(clean)}行 / "
          f"rejected {len(rejected)}行 ({rate:.1f}%)")
    return clean, rejected


# ---------------- 03章: 名寄せと結合 ----------------
def load_masters():
    alias = pd.read_csv("/data/shop_alias.csv", dtype=str, keep_default_na=False)
    alias["alias"] = alias["alias"].map(norm)

    shops = pd.read_csv("/data/shops.csv", dtype=str, keep_default_na=False)
    shops = shops.replace("", pd.NA)
    shops["close_date"] = pd.to_datetime(shops["close_date"])

    items = pd.read_csv("/data/items.csv", dtype=str, keep_default_na=False)
    items = items.replace("", pd.NA)
    items["discontinued_date"] = pd.to_datetime(items["discontinued_date"])
    return alias, shops, items


def join_master(clean):
    """clean にマスタを結合する。行は増やさない。除外もしない。"""
    alias, shops, items = load_masters()

    df = clean.rename(columns={"shop_name": "shop_name_raw"})

    df = df.merge(alias, left_on="shop_name_raw", right_on="alias",
                  how="left", validate="m:1").drop(columns="alias")
    df = df.merge(shops[["shop_cd", "shop_name", "area", "close_date"]],
                  on="shop_cd", how="left", validate="m:1")
    df = df.merge(items[["item_cd", "item_name", "category", "discontinued_date"]],
                  on="item_cd", how="left", validate="m:1")

    df["category"] = df["category"].fillna("未分類")

    assert len(df) == len(clean), "結合で行数が変わりました"

    no_shop = df["shop_cd"].isna().sum()
    no_item = df["item_name"].isna().sum()
    closed = (df["close_date"].notna() & (df["sale_date"] > df["close_date"])).sum()
    print(f"結合: {len(df)}行  店舗未マッチ {no_shop} / 商品未マッチ {no_item} / "
          f"閉店後の売上 {closed}")
    return df


raw = load_raw()
clean, rejected = clean_raw(raw)
fact = join_master(clean)
fact.head(3)

---
## 1. 粒度を決める

集計の前に決めることが1つあります。**サマリの1行が何を表すか**です。
これを**粒度(グレイン)**と呼びます。

粒度が決まっていないと、こういう会話になります。

> 「売上の表を作って」
> 「日ごと? 月ごと?」
> 「日ごとで」
> 「店ごとに分ける?」
> 「分けて」
> 「じゃあ商品も分ける?」…

**先に決めて、書いておきます。** この教材ではこうします。

```
1行 = 1日 × 1店舗 × 1カテゴリ
```

### 粒度が変わると、何が変わるか

同じデータでも、まとめ方を変えると行数が変わります。

In [ ]:
for keys in [["shop_cd"],
             ["sale_date"],
             ["sale_date", "shop_cd"],
             ["sale_date", "shop_cd", "category"],
             ["sale_date", "shop_cd", "item_cd"]]:
    k = [fact["sale_date"].dt.date if c == "sale_date" else c for c in keys]
    n = fact.groupby(k, observed=True).ngroups
    print(f"{n:3}行   {' × '.join(keys)}")

細かくするほど行が増えます。**細かいほうが良い、ということではありません。**

| 粒度 | 向いていること | 向かないこと |
| --- | --- | --- |
| 粗い(店舗だけ) | ひと目で分かる | 「いつ落ちたのか」が見えない |
| 細かい(商品まで) | 何でも追える | 行が多くて読めない。数字が細かすぎてぶれる |

**「誰が何を見たいのか」から決めます。** 店長が毎朝見る表なら、
日次 × 店舗 × カテゴリくらいが読める上限です。

> 粒度を細かくしておけば、あとから粗くするのは足し算だけでできます。
> 逆はできません。**迷ったら、必要な範囲でいちばん細かい粒度**を選んでおくと、
> あとで困りにくくなります。ただし「必要な範囲で」です。

In [ ]:
# ✍ 書いてみる: 「日付 × 店舗 × 商品」の粒度にすると何行になるか数えてください。
#              (ヒント: 上のセルと同じ形。ngroups が使えます)

ans = ...   # ここに書く

assert ans == 31, f"31行のはずです: {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = fact.groupby([fact["sale_date"].dt.date, "shop_cd", "item_cd"],
                   observed=True).ngroups
```

</details>

**31行**です。`fact` は34行なので、**3行ぶん減りました。**

いちばん細かい粒度まで割ったのに、行数が元より減っています。
理由は03章の7節で見たとおりです。

In [ ]:
g = fact.groupby([fact["sale_date"].dt.date, "shop_cd", "item_cd"], observed=True)

print("グループ数:", g.ngroups, " 元の行数:", len(fact))
print()
print("2行以上あるグループ:")
print(g.size()[g.size() > 1])

**3グループが2行ずつ**持っています。03章で「店頭とECの両方で売れた」と確認した3組です。

34 − 3 = 31。数が合いました。

ここで起きているのは、**重複排除ではなく足し合わせ**です。
4/2 の渋谷店のコーヒーは、店頭で3個・EC で1個売れました。
集計すると1行になり、数量4個・1,800円になります。**どちらの売上も消えていません。**

> 03章で「この重複は消さない」と決めたのは、こういう扱いになるからです。
> 明細として持っておけば、集計の段階で自然に合算されます。
> 先に消していたら、片方の売上が失われていました。

---
## 2. 何を集計に入れるかを決める (fact → target)

ここが `fact` と `target` の分かれ目です。

### 2-1. テスト伝票

旧POSに、備考が `テスト` の行があります。

In [ ]:
test = fact[fact["note"].fillna("") == "テスト"]
test[["sale_date", "shop_name", "item_name", "qty", "amount_incl", "source"]]

4/8 に渋谷店で、コーヒー2個・900円。備考に `テスト` と書いてあります。

**これは実際の売上ではありません。** 端末の動作確認で打った伝票です。
売上に入れると、900円が水増しされます。

> こういう行が本番データに混ざるのは、珍しいことではありません。
> 備考に `テスト` と書いてあるだけまだ親切なほうで、
> 何の印も無いこともあります。そうなると見つける方法がありません。
> **「印が付いているものは、必ず拾う」**のが現実的な線です。

### 2-2. 閉店した店の売上

03章で見つけた、大宮店(S04)の4/10・450円です。

In [ ]:
closed = fact[fact["close_date"].notna() & (fact["sale_date"] > fact["close_date"])]
closed[["sale_date", "shop_name", "close_date", "item_name", "amount_incl", "source"]]

**この450円をどう扱うかは、判断です。** 2つの立場があります。

| 立場 | 言い分 |
| --- | --- |
| 除外する | 閉店した店の売上は、店舗別サマリに置き場所が無い。3店舗の表に4店舗目が現れると読む人が混乱する |
| 残す | 売上が立った事実は消せない。全社合計が450円合わなくなる |

**この教材では除外します。** サマリは「営業中の3店舗を比べるための表」だからです。

ただし、**除外したことを黙って隠しません。** 次の2つをやります。

- 除外した**件数と金額を毎回出力する**
- `fact` は除外前のまま残す(調べたくなったら戻れる)

> 実務では、こういう行を `out/excluded/` のような場所に**書き出しておく**こともあります。
> 「サマリからは外したが、消してはいない」という状態を作るためです。
> 02章で `rejected` を隔離したのと同じ考え方です。

### 2-3. 返品は除外しません

EC に返品が2行あります。数量も金額もマイナスです。

In [ ]:
ret = fact[fact["note"].fillna("") == "返品"]
print(ret[["sale_date", "shop_name", "item_name", "qty", "amount_incl"]])
print()
print(f"返品の金額合計: {ret['amount_incl'].sum():,}円")

**-950円です。これは除外しません。**

返品は「間違ったデータ」ではなく、**起きた事実**です。
売上から差し引かれるのが正しい姿なので、マイナスのまま合計に入れます。

「マイナスがあると見づらい」と思って除外すると、**売上が実態より大きく出ます。**
これは静かに間違った数字を出すことになるので、してはいけません。

> 除外の判断で迷ったときの目安です。
>
> - **業務として起きた事実か** → 残す(返品、値引き、キャンセル)
> - **データの都合で混ざったものか** → 除外する(テスト伝票、閉店店舗、二重送信)
>
> 「見づらいから」は理由になりません。

### 2-4. マスタに無い商品(`0099`)は残します

03章で `未分類` に寄せた4行・4,059円です。**これも除外しません。**

売れたことは確かで、金額も分かっています。分からないのはカテゴリだけです。
`未分類` として表に出しておけば、見た人が「これは何?」と聞いてくれます。

**除外すると、聞かれる機会そのものが消えます。**

### 2-5. 除外を数えてから落とす

決まりました。落とすのは**テスト伝票**と**閉店後の売上**の2種類です。

書き方に1つだけ約束を作ります。**数えてから落とします。**

In [ ]:
is_test = fact["note"].fillna("") == "テスト"
is_closed = fact["close_date"].notna() & (fact["sale_date"] > fact["close_date"])
drop = is_test | is_closed

print(f"テスト伝票 {is_test.sum()}行  {fact.loc[is_test, 'amount_incl'].sum():,}円")
print(f"閉店後     {is_closed.sum()}行  {fact.loc[is_closed, 'amount_incl'].sum():,}円")
print()

target = fact[~drop].reset_index(drop=True)

print(f"fact   {len(fact)}行  {fact['amount_incl'].sum():,}円")
print(f"target {len(target)}行  {target['amount_incl'].sum():,}円")

```
fact   34行  26,489円
target 32行  25,139円
```

**26,489 − 900 − 450 = 25,139。** 引き算が合っています。

`fact[~drop]` と書いて、`fact` そのものは書き換えていないところに注目してください。
**元の表は残ります。** あとで「除外した450円は何だったか」を調べたくなったら、
`fact[drop]` で取り出せます。

> `fact.drop(fact[drop].index, inplace=True)` のように元を書き換えると、
> **もう戻れません。** ノートブックでは上のセルから流し直すことになります。
> **元を残して新しい名前を付ける**、という書き方にしておくと、やり直しが楽です。

In [ ]:
# ✍ 書いてみる: 除外した2行の金額合計を、fact と target の差として求めてください。

ans = ...   # ここに書く

assert ans == 1350, f"1350円のはずです: {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = fact["amount_incl"].sum() - target["amount_incl"].sum()
```

</details>

**1,350円**(テスト900 + 閉店450)です。

この引き算が合わないときは、除外の条件が思ったとおりに効いていません。
**除外を書いたら、必ず差を検算してください。**

---
## 3. 集計する

粒度も対象も決まりました。まとめます。

### 3-1. 日付から「日」を取り出す

`sale_date` は `datetime64` なので、時刻まで持てる型です。
日単位でまとめたいので、日付だけ取り出します。

In [ ]:
day = target["sale_date"].dt.date.rename("sale_date")

print(day.head(3).tolist())
print("種類:", day.nunique(), "日ぶん")

`.dt.date` で `datetime.date` になります。`rename` しているのは、
**このあと `groupby` のキーに使うと、そのまま列名になる**からです。
名前を付けておかないと `level_0` のような列名になります。

> 今回のデータに時刻は入っていないので、`.dt.date` を通さなくても同じ結果になります。
> それでも通しておくのは、**時刻付きのデータが来たときに壊れないため**です。
> 時刻が入ると `2024-04-01 09:15` と `2024-04-01 14:30` が別の日として集計されます。

`nunique()` が **9** なのを覚えておいてください。4/1〜4/10 は10日あります。
**1日足りません。** これは4節で扱います。

### 3-2. `groupby` と `agg`

複数の集計を一度に書くときは、**名前付き集計**が読みやすいです。

```python
.agg(出したい列名=("元の列", "集計方法"))
```

In [ ]:
by_category = (target
               .groupby([day, "shop_cd", "category"])
               .agg(qty=("qty", "sum"),
                    amount_jpy=("amount_incl", "sum"),
                    n_lines=("item_cd", "size"))
               .reset_index())

print(by_category.shape)
by_category.head(10)

**28行**になりました。列は決めた名前で出ています。

| 列 | 中身 |
| --- | --- |
| `qty` | 数量の合計 |
| `amount_jpy` | 金額の合計(税込) |
| `n_lines` | **何明細をまとめた行か** |

`n_lines` を入れているのには理由があります。
「1行あたりの金額が異常に大きい」と思ったとき、
**それが1明細なのか10明細の合計なのかで、調べる方向が変わります。**

> `"size"` と `"count"` は違います。`"count"` は**欠損を除いて数える**ので、
> 欠損があると明細数より小さくなります。
> **「何行あったか」を知りたいときは `"size"`** です。

`amount_incl` を `amount_jpy` に改名しているのは、
**サマリを見る人は「税込か税抜か」を意識しない**からです。
出口では、内部の都合が名前に出ないようにします。

In [ ]:
# ✍ 書いてみる: 同じやり方で、カテゴリ別 (日付も店舗も使わない) の売上合計を出してください。
#              列名は amount_jpy にします。

ans = ...   # ここに書く

assert ans.set_index("category")["amount_jpy"].to_dict() == {
    "デザート": 7579, "未分類": 4059, "軽食": 3850, "飲料": 9651}, ans
print("OK")
print(ans)

<details>
<summary>答え</summary>

```python
ans = (target.groupby("category")
       .agg(amount_jpy=("amount_incl", "sum"))
       .reset_index())
```

</details>

`未分類` が 4,059円で、全体の16%です。**表に出しておいてよかった**大きさです。
黙って落としていたら、6分の1が消えていました。

---
## 4. 売上ゼロの日を埋める

3-1 で「日付が9種類しかない」ことに気づきました。確かめます。

In [ ]:
print(sorted(day.unique()))

**4/7 がありません。** 日曜日で、全店休みだったのかもしれません。

同じことが店舗単位でも起きています。

In [ ]:
by_shop_raw = (target
               .groupby([day, "shop_cd"])
               .agg(qty=("qty", "sum"),
                    amount_jpy=("amount_incl", "sum"),
                    n_lines=("item_cd", "size")))

print(f"実際にある組み合わせ: {len(by_shop_raw)}")
print(f"あるべき組み合わせ  : 10日 × 3店 = 30")
by_shop_raw.head()

**22組しかありません。** 8組ぶん、行そのものが存在しません。

### 4-1. 「行が無い」と「ゼロ」は違います

このまま出すと、受け取った側で困ることが起きます。

| やりたいこと | 行が無いと |
| --- | --- |
| 折れ線グラフを描く | 4/7 が飛ばされて、4/6 と 4/8 が直接つながる。**落ち込みが見えない** |
| 前日比を出す | 前日の行が無いので計算できない |
| 平均を出す | 分母が22になる。30で割るべきなのに |
| 「売れなかった日」を数える | **数えられない。行が無いので** |

いちばん困るのは最後です。
**「売上ゼロ」という情報は、行が無いことでは伝わりません。**
見る側からは「ゼロだった」のか「データが届いていない」のか区別が付きません。

### 4-2. あるべき組み合わせを作って、そこに流し込む

考え方はこうです。

```
あるべき組み合わせ全部 (日付 × 店舗) を作る
   ↓
そこへ集計結果を流し込む (reindex)
   ↓
流し込まれなかったところを 0 で埋める
```

In [ ]:
shops_open = sorted(
    load_masters()[1].pipe(lambda s: s.loc[s["close_date"].isna(), "shop_cd"]))
print("営業中の店:", shops_open)

grid = pd.MultiIndex.from_product(
    [pd.date_range(day.min(), day.max(), freq="D").date, shops_open],
    names=["sale_date", "shop_cd"])

print("格子の大きさ:", len(grid))

店舗の一覧を**マスタから作っている**のがポイントです。

`target` から作ってはいけません。**1か月まったく売れなかった店は、
`target` に1行も現れない**からです。それでは埋める意味がありません。

`close_date` が入っていない店だけを取っているので、大宮店(S04)は入りません。
2-2 で「S04 を除外する」と決めたことと、ここで「S04 を格子に入れない」ことは、
**同じ1つの判断**です。片方だけ直すと表が壊れるので、
最後は関数にまとめて1箇所にします(6節)。

In [ ]:
by_shop = by_shop_raw.reindex(grid, fill_value=0).reset_index()

print(by_shop.shape)
print(f"売上ゼロの行: {(by_shop['amount_jpy'] == 0).sum()}")
by_shop.head(9)

**30行になり、うち8行がゼロ**です。4/1 の新宿店(S02)がゼロで入っているのが見えます。

`reindex` は「この並びに合わせ直す」という操作です。
- 格子にあって集計に無い組み合わせ → `fill_value=0` で埋まる
- 集計にあって格子に無い組み合わせ → **消えます**

2つ目に注意してください。`shops_open` に入れ忘れた店があると、
**その店の売上が黙って消えます。** `reindex` を使ったら、
**合計が変わっていないか必ず検算してください**(5節でやります)。

### 4-3. どこまで埋めるかは、自分で決めます

上のセルでは `day.min()` から `day.max()`、つまり **4/1〜4/10** を埋めました。
**4月末まで埋めていません。** これは意図的です。

In [ ]:
month_end = pd.MultiIndex.from_product(
    [pd.date_range("2024-04-01", "2024-04-30", freq="D").date, shops_open],
    names=["sale_date", "shop_cd"])
wide = by_shop_raw.reindex(month_end, fill_value=0)

print(f"データのある範囲だけ: {len(grid)}行 (ゼロ {(by_shop['amount_jpy'] == 0).sum()}行)")
print(f"月末まで埋める     : {len(wide)}行 (ゼロ {(wide['amount_jpy'] == 0).sum()}行)")

月末まで埋めると90行になり、**68行がゼロ**になります。

けれど 4/11 以降は「売れなかった日」ではありません。
**まだデータが届いていない日**です。この2つを同じ `0` にしてしまうと、
「4月は売上が激減した」という**嘘の表**ができあがります。

```
売上がゼロだった      →  0 で埋めてよい
データが届いていない   →  0 で埋めてはいけない
```

見た目は同じ `0` ですが、意味は正反対です。

**埋めてよいのは「データが揃っていると言える範囲」だけ**です。
今回は届いたデータの範囲(4/1〜4/10)を、そのまま「揃っている範囲」と見なしました。

> 実務では「対象期間」を外から渡します(`--month 2024-04` のように)。
> そのうえで、月の途中で流すなら実行日まで、月が締まっているなら月末まで、
> と埋める範囲を決めます。**日付の範囲は、データではなく運用が決めます。**
> これは05章の `run(month)` につながる話です。

### 4-4. カテゴリまでは埋めません

日付 × 店舗は埋めました。**カテゴリまで埋めるとどうなるか**も見ておきます。

In [ ]:
n_cat = target["category"].nunique()
print(f"日付 × 店舗            = {len(grid)}行")
print(f"日付 × 店舗 × カテゴリ  = {len(grid)} × {n_cat} = {len(grid) * n_cat}行")
print(f"  実際に売上がある行     = {len(by_category)}行")
print(f"  ゼロで埋まる行         = {len(grid) * n_cat - len(by_category)}行")

**120行のうち92行がゼロ**になります。表の8割が `0` です。

「渋谷店の4/7のデザートの売上は0円」という行に、意味はありません。
**その日は店ごと閉まっている**からです。同じことが92回書いてあるだけの表になります。

判断はこうです。

| | 埋める? | なぜ |
| --- | --- | --- |
| 日次 × 店舗 | **埋める** | 「その日、その店が売れなかった」は意味のある情報 |
| 日次 × 店舗 × カテゴリ | **埋めない** | 組み合わせが多すぎて、ゼロの意味が薄まる |

だから**サマリを2つ出します。**

- `by_shop` … 日次 × 店舗。**ゼロを埋めた30行。**グラフや前日比に使う
- `by_category` … 日次 × 店舗 × カテゴリ。**売れたものだけ28行。**内訳を見るのに使う

**1つの表に全部を詰め込もうとしない**、というのがここでの学びです。
用途が違うなら、表も分けます。

In [ ]:
# ✍ 書いてみる: by_shop から、売上がゼロだった日と店舗の組み合わせを取り出してください。

ans = ...   # ここに書く

assert len(ans) == 8, f"8行のはずです: {len(ans)}"
assert ans["shop_cd"].value_counts().to_dict() == {"S02": 4, "S03": 3, "S01": 1}, \
    ans["shop_cd"].value_counts().to_dict()
print("OK")
print(ans[["sale_date", "shop_cd", "amount_jpy"]])

<details>
<summary>答え</summary>

```python
ans = by_shop[by_shop["amount_jpy"] == 0]
```

</details>

4/7 は3店とも売上ゼロです。**全店同じ日に休み**なので、
「日曜は休業」という営業の実態が、この表から読み取れます。

もう1つ読み取れることがあります。**ゼロの日が S02(新宿店)に4日**と偏っています。
残る3日は S03(横浜店)、S01(渋谷店)は4/7の1日だけです。
「新宿店は10日のうち4日、1件も売れていない」というのは、
**気にしたほうがよい数字**です。データが届いていないのかもしれません。

**どちらも、ゼロを埋めたから読み取れた**ことです。行が無いままだったら、
4/7 という日付自体が表に現れず、新宿店の4日も数えられませんでした。

---
## 5. 検算する

集計はここまでです。**出す前に、必ず検算します。**

集計は「合っていそうに見えるが間違っている」ことが起きやすい処理です。
`groupby` のキーを1つ書き忘れても、`reindex` で行が落ちても、**エラーは出ません。**

### 5-1. 合計を突き合わせる

いちばん効くのがこれです。**元と集計後で、合計が一致するか。**

In [ ]:
total = target["amount_incl"].sum()

print(f"target      {total:,}円")
print(f"by_shop     {by_shop['amount_jpy'].sum():,}円")
print(f"by_category {by_category['amount_jpy'].sum():,}円")

assert by_shop["amount_jpy"].sum() == total
assert by_category["amount_jpy"].sum() == total
print("\n3つとも一致しました")

**25,139円で一致**しました。

この `assert` が、4節で書いた「`reindex` で行が消えても気づけない」への対策です。
店舗を1つ入れ忘れていたら、ここで止まります。

### 5-2. 上流までさかのぼって突き合わせる

もう1段さかのぼります。`fact` からの引き算が合うかどうかです。

In [ ]:
excluded = fact["amount_incl"].sum() - total

print(f"fact         {fact['amount_incl'].sum():,}円")
print(f"  除外        -{excluded:,}円  (テスト900 + 閉店450)")
print(f"target       {total:,}円")
print()
print(f"clean        {clean['amount_incl'].sum():,}円   ← 02章")
print(f"rejected     {rejected['amount_incl'].sum(skipna=True):,}円   ← 隔離した2行(金額があるのは1行)")

assert excluded == 1350
assert fact["amount_incl"].sum() == clean["amount_incl"].sum()
print("\n02章からの引き算が合っています")

```
clean   26,489円      02章で整えた34行
  ↓ 結合 (増減なし)
fact    26,489円      03章
  ↓ 除外 -1,350円
target  25,139円      04章 ← いまここ
```

**02章から04章まで、金額の増減がすべて説明できます。**
これが、02章の最後に「数字を1つ握って次の章に行く」と言った理由です。

「なんとなく減った」を1度でも通すと、あとから原因を探すのが非常に難しくなります。

### 5-3. 手で数えて確かめる

最後にもう1つ。**小さいところを1つ選んで、手で数えます。**

このデータが36行しかないのは、これができるようにするためです。
4/1 の売上を、元のCSVから直接数えてみます。

In [ ]:
print("--- 元のCSVにある 4/1 の行 ---")
print("旧POS: 2024年4月1日 ミナトストア 渋谷 コーヒー ２個 ９００円")
print("旧POS: 2024年4月1日 ミナトストア 渋谷 ケーキ  １個 ６００円")
print("新POS: 2024/4/1     みなとストア横浜店 コーヒー 2個 900円")
print("新POS: 2024/4/1     みなとストア横浜店 紅茶    1個 400円")
print()
print("--- 手で数えた期待値 ---")
print("S01 (渋谷) = 900 + 600 = 1500")
print("S02 (新宿) =                0   ← 4/1 は売上なし")
print("S03 (横浜) = 900 + 400 = 1300")
print()
print("--- サマリの値 ---")
print(by_shop[by_shop["sale_date"] == pd.Timestamp("2024-04-01").date()])

**一致しました。** S01 が1500、S02 が0、S03 が1300 です。

たった1日ぶんですが、この確認が通ると

- 文字コードの変換(旧POSの全角)
- 名寄せ(`ミナトストア 渋谷` → S01)
- 型変換(`９００` → 900)
- 集計
- ゼロ埋め(S02)

の**全部が正しく通ったこと**が確かめられます。
パイプラインの端から端までを、1本の線で検証したことになります。

> 実務のデータは手で数えられません。それでも、
> **「1件だけ選んで、生データから出力まで追いかける」**ことはできます。
> これを**トレース**と呼びます。新しいパイプラインを作ったら、必ず1回はやってください。
> 集計結果を眺めているだけでは見つからない間違いが、これで見つかります。

In [ ]:
# ✍ 書いてみる: 4/9 の売上合計 (3店舗ぶん) を by_shop から求めてください。

ans = ...   # ここに書く

assert ans == 4819, f"4819円のはずです: {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = by_shop.loc[by_shop["sale_date"] == pd.Timestamp("2024-04-09").date(),
                  "amount_jpy"].sum()
```

</details>

**4,819円**で、4月でいちばん売れた日です。

末尾が `9` という半端な数になっているのは、02章で見た**税の丸め**のせいです。
EC の 728円(税抜)を税込にした 801円 が入っています。
定価から計算すると 800円なので、1円多く出ています。

**由来の分かっているズレ**なので、このまま進めます。
分からないズレとの違いは、**説明が付くかどうか**だけです。

---
## 6. この章のまとめ

除外と集計を**関数にまとめます**。2つに分けます。

In [ ]:
# ---------------- 04章: 集計 ----------------
AGG = {"qty": ("qty", "sum"),
       "amount_jpy": ("amount_incl", "sum"),
       "n_lines": ("item_cd", "size")}


def open_shop_cds():
    """営業中の店舗コード。閉店した店は格子に含めない。"""
    _, shops, _ = load_masters()
    return sorted(shops.loc[shops["close_date"].isna(), "shop_cd"])


def select_target(fact):
    """fact から集計対象外を除いて target にする。除外は必ず数える。"""
    is_test = fact["note"].fillna("") == "テスト"
    is_closed = fact["close_date"].notna() & (fact["sale_date"] > fact["close_date"])
    drop = is_test | is_closed

    print(f"除外: テスト伝票 {is_test.sum()}行 "
          f"({fact.loc[is_test, 'amount_incl'].sum():,}円) / "
          f"閉店後 {is_closed.sum()}行 "
          f"({fact.loc[is_closed, 'amount_incl'].sum():,}円)")
    target = fact[~drop].reset_index(drop=True)
    print(f"集計対象: fact {len(fact)}行 → target {len(target)}行  "
          f"{target['amount_incl'].sum():,}円")
    return target


def summarize(fact):
    """fact (除外前) を受け取り、(店舗別, カテゴリ別) の日次サマリを返す。

    除外はこの関数の中でやる。渡すのは fact であって target ではない。
    """
    target = select_target(fact)
    day = target["sale_date"].dt.date.rename("sale_date")

    by_category = (target.groupby([day, "shop_cd", "category"])
                   .agg(**AGG).reset_index())

    # 売上ゼロの日を埋める。埋めてよいのはデータが揃っている範囲だけ
    grid = pd.MultiIndex.from_product(
        [pd.date_range(day.min(), day.max(), freq="D").date, open_shop_cds()],
        names=["sale_date", "shop_cd"])
    by_shop = (target.groupby([day, "shop_cd"])
               .agg(**AGG).reindex(grid, fill_value=0).reset_index())

    total = target["amount_incl"].sum()
    assert by_category["amount_jpy"].sum() == total, "カテゴリ別の合計が target と合いません"
    assert by_shop["amount_jpy"].sum() == total, "店舗別の合計が target と合いません"

    print(f"サマリ: 店舗別 {len(by_shop)}行 "
          f"(うち売上ゼロ {(by_shop['amount_jpy'] == 0).sum()}行) / "
          f"カテゴリ別 {len(by_category)}行  合計 {total:,}円")
    return by_shop, by_category


by_shop, by_category = summarize(fact)
by_shop.head()

**`summarize` が受け取るのは `fact`(除外前)です。`target` ではありません。**

除外を関数の外に出して `summarize(target)` と書くこともできますが、そうしません。
外に出すと、**呼ぶ側が除外を忘れられる**ようになります。

```python
summarize(target)   # 呼ぶ側が select_target を忘れたら、除外されないまま集計される
summarize(fact)     # 忘れようがない
```

**忘れると壊れることは、忘れられない場所に置きます。**

同じ理由で、`assert` も関数の中に入れてあります。
5節でやった検算は「1回やって安心するもの」ではなく、
**毎回の実行で通り続けるべき条件**だからです。

In [ ]:
# ✍ 書いてみる: summarize が返した2つの表の合計が、どちらも 25,139 円か確かめて、
#              True を ans に入れてください。

ans = ...   # ここに書く

assert ans is True
print("OK")

<details>
<summary>答え</summary>

```python
ans = bool(
    by_shop["amount_jpy"].sum() == 25139
    and by_category["amount_jpy"].sum() == 25139
)
```

</details>

### できあがった表

In [ ]:
print("=== by_shop (日次 × 店舗。ゼロ埋めあり) ===")
print(by_shop.pivot(index="sale_date", columns="shop_cd", values="amount_jpy"))
print()
print("=== by_category の内訳 ===")
print(by_category.groupby("category")["amount_jpy"].sum())

`pivot` で縦横に並べ替えると、**日次サマリらしい見た目**になります。

4/6 の S01 がマイナス(返品だけの日)、4/7 が全店ゼロ(休業)、
というのが表から読み取れます。**数字を並べただけで、営業の様子が見えてきます。**

これがサマリを作る目的です。36行の明細を眺めていても、この形は見えません。

---
## この章で分かったこと

| | |
| --- | --- |
| 粒度 | サマリの1行が何かを**先に決める**。細かければ良いわけではない |
| 除外の基準 | **業務として起きた事実は残す**(返品)。**データの都合で混ざったものは落とす**(テスト・閉店) |
| 除外の書き方 | **数えてから落とす。** 元の表は書き換えず、新しい名前を付ける |
| 集計 | 名前付き集計 `agg(名前=("列", "方法"))`。`"size"` と `"count"` は違う |
| `n_lines` | 何明細をまとめた行かを残す。異常値を追うときに効く |
| ゼロ埋め | 「行が無い」と「ゼロ」は違う。**格子は元データではなくマスタから作る** |
| 埋める範囲 | **データが揃っている範囲だけ。** 未着の日をゼロにすると嘘になる |
| 表を分ける | 用途が違うなら表も分ける。1つに詰め込まない |
| 検算 | 合計の突き合わせ / 上流からの引き算 / **1件を手で追うトレース** |
| 関数の形 | 忘れると壊れることは、関数の中に入れる |

## 次の章に持ち越す宿題

サマリはできました。あとは**出して、また流せるようにする**だけです。

- [ ] Parquet で書く(型を保ったまま保存する)
- [ ] 出力にスキーマを強制する(取りこぼした汚れを、止まる障害に変える)
- [ ] **同じ月を2回流しても、結果が変わらないようにする**
- [ ] 02章で隔離した2行の**訂正が届いたら**、4月を作り直す
- [ ] 5月ぶんを流したときに、4月に触らないようにする

次: `05-output.ipynb`